# CENG 467 — Train mT5-small + LoRA on Synthetic Turkish Summaries

Colab T4 (free tier). Mounts Drive, clones repo, installs deps, runs the training pipeline end to end.


## 1. Mount Google Drive (optional but strongly recommended for checkpoints)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib, os
WORK = pathlib.Path('/content/drive/MyDrive/ceng467_termproject')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print('cwd =', os.getcwd())


## 2. Clone the repository (replace with your fork URL)


In [ ]:
REPO_URL = 'https://github.com/cagancaliskan/turkish-summarization-distillation.git'
if not pathlib.Path('turkish-summarization-distillation').exists():
    !git clone $REPO_URL
os.chdir('turkish-summarization-distillation')
!ls


## 3. Install pinned dependencies


In [ ]:
!pip install -q -r requirements.txt


## 4. Provide API keys
Either upload your `.env` to the repo root, or set them inline below.


In [ ]:
import os
os.environ['OPENAI_API_KEY']    = os.environ.get('OPENAI_API_KEY')    or 'sk-...'
os.environ['ANTHROPIC_API_KEY'] = os.environ.get('ANTHROPIC_API_KEY') or 'sk-ant-...'


## 5. Download data


In [ ]:
!bash scripts/01_download_data.sh


## 6. Generate teacher summaries
Each call below is cached on disk; re-running is free.


In [ ]:
!bash scripts/02_generate_teacher.sh openai concise 10000
!bash scripts/02_generate_teacher.sh anthropic concise 10000


## 7. Train the student variants


In [ ]:
!bash scripts/03_train_student.sh --teacher openai    --prompt concise --size 10000 --lora-rank 8
!bash scripts/03_train_student.sh --teacher anthropic --prompt concise --size 10000 --lora-rank 8
!bash scripts/03_train_student.sh --teacher human     --prompt concise --size 10000 --lora-rank 8


## 8. Quick smoke test of inference


In [ ]:
!python -m src.student.infer \
  --model-path outputs/checkpoints/openai_concise_n10000_r8/final \
  --input data/raw/mlsum_tr/test.jsonl \
  --out outputs/predictions/_smoke_S_gpt.jsonl \
  --limit 16
!head -n 1 outputs/predictions/_smoke_S_gpt.jsonl
